In [2]:
import os
import re
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from plotnine import *
from plotnine import ggplot, aes, theme_minimal, labs
from plotnine import geom_point, scale_size, scale_size_continuous
from plotnine import scale_x_discrete
from geopy.distance import geodesic
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon, LineString, MultiLineString
from scipy.stats import ttest_ind

In [3]:
# The Goal of this script is to plot the number of start or end points in each location for each month of the outbreak

In [15]:
# Upload updated MCC.csv

mcc_updt = pd.read_csv("./SERAPHIM/nov2021-2024/NAm_wild-aves_nov2021-2024_ha_MCC_updated.csv", sep =',')

In [26]:
# Loop through each month_year to get counts per state

list_of_months = ["2021_september", "2021_october","2021_november","2021_december", "2022_january", "2022_february", "2022_march", "2022_april", "2022_may", "2022_june",
               "2022_july", "2022_august", "2022_september", "2022_october", "2022_november", "2022_december","2023_january", "2023_february", "2023_march", "2023_april", 
               "2023_may", "2023_june", "2023_july", "2023_august", "2023_september", "2023_october", "2023_november", "2023_december", "2024_january", "2024_february", 
               "2024_march", "2024_april", "2024_may", "2024_june","2024_july", "2024_august", "2024_september", "2024_october", "2024_november", "2024_december"]

months = list_of_months

for month in months:

    mcc_month = mcc_updt[mcc_updt["month_year"] == month]

    # Count startStates
    start_counts = mcc_month["startState"].value_counts().astype(str).reset_index()
    start_counts.columns = ["state", "startState_counts"]

    # Count endStates
    end_counts = mcc_month["endState"].value_counts().astype(str).reset_index()
    end_counts.columns = ["state", "endState_counts"]

    # Merge both counts on 'state'
    month_counts = pd.merge(start_counts, end_counts, on="state", how="outer")

    # Add the month info
    month_counts["month_year"] = month

    # Optional: reorder columns
    month_counts = month_counts[["month_year", "state", "startState_counts", "endState_counts"]]
    
    month_counts["total_counts"] = month_counts["startState_counts"].astype(float) + month_counts["endState_counts"].astype(float)

    output_name = "./SERAPHIM/nov2021-2024/" + month + "counts.csv"

    month_counts.to_csv(output_name, sep = ',')

In [38]:
# Import lat/longs for each division

state_latlongs = pd.read_csv("./metadata/helpful_lists/flywayregions_biggerlist.csv", sep = ',')

In [39]:
# Merge counts and coordinates
# bubble_data = pd.merge(finalmeta_ha_state_counts_filtered, state_latlongs, on='location', how='left')
bubble_data = pd.merge(metadata_state_counts_filtered, state_latlongs, on='location', how='left')

In [27]:
# Need a shapefile to dictate polygon boundaries

# North America shapefile
NAm_shp_path = "./metadata/shapefiles/north_america/north_america.shp"
NAm_gdf = gpd.read_file(NAm_shp_path)

# Downloaded US shapefile from Natural Earth
# Downloaded Canadian province shapefile from: https://open.canada.ca/data/en/dataset/a883eb14-0c0e-45c4-b8c4-b54c4a819edb/resource/12c03de6-c3f7-4f5f-bb5c-d479f2332842
# Downloaded Mexican state shapefile from: https://github.com/jschleuss/mexican-states

In [28]:
# Filter GDF and set the CRS 

NAm_gdf["geometry_name"] = NAm_gdf["name"].combine_first(NAm_gdf["PRNAME"])
columns = ["geometry", "geometry_name", "place", "type", "latitude", "longitude", "fips", "region", "gadm_level"]
NAm_gdf = NAm_gdf.loc[0:, columns]

NAm_gdf = NAm_gdf.to_crs("EPSG:4326")

In [38]:
# Create a DataFrame of lat/lon from the geometries

def geom_to_df(geom, region_name):
    if isinstance(geom, Polygon):
        coords = list(geom.exterior.coords)
        df = pd.DataFrame(coords, columns=['lon', 'lat'])
        df['region'] = region_name
        df['group'] = f"{region_name}_0"
        return df

    elif isinstance(geom, MultiPolygon):
        dfs = []
        for i, part in enumerate(geom.geoms):
            coords = list(part.exterior.coords)
            df = pd.DataFrame(coords, columns=['lon', 'lat'])
            df['region'] = region_name
            df['group'] = f"{region_name}_{i}"
            dfs.append(df)
            # Add NaN to separate parts (important for line-based plotting)
            dfs.append(pd.DataFrame({
                'lon': [np.nan],
                'lat': [np.nan],
                'region': [region_name],
                'group': [f"{region_name}_{i}"]
            }))
        return pd.concat(dfs, ignore_index=True)

    if isinstance(geom, LineString):
        coords = list(geom.coords)
        df = pd.DataFrame(coords, columns=['lon', 'lat'])
        df['region'] = region_name
        df['group'] = f"{region_name}_0"
        return df

    elif isinstance(geom, MultiLineString):
        dfs = []
        for i, part in enumerate(geom.geoms):
            coords = list(part.coords)
            df = pd.DataFrame(coords, columns=['lon', 'lat'])
            df['region'] = region_name
            df['group'] = f"{region_name}_{i}"
            dfs.append(df)
#            dfs.append(pd.DataFrame({'lon': [np.nan],'lat': [np.nan],'region': [region_name],'group': [f"{region_name}_{i}"]}))
        return pd.concat(dfs, ignore_index=True)

    else:
        return pd.DataFrame(columns=['lon', 'lat', 'region', 'group'])

In [39]:
map_df = pd.concat([geom_to_df(row.geometry, row['featurecla']) for idx, row in coastlines_gdf.iterrows()], ignore_index=True)
filtered_map_df = map_df[
    (map_df['lat'] >= 0) & (map_df['lat'] <= 80) &
    (map_df['lon'] >= -180) & (map_df['lon'] <= -30)
]

starts = (
    ggplot(filtered_map_df) +
    geom_path(aes(x='lon', y='lat', group='group'), color='gray') +
    geom_point(data=unknown_starts, mapping=aes(x='startLon', y='startLat'), color='blue', alpha=0.6) +
    coord_fixed() +
    theme_minimal() +
    theme(figure_size=(7, 5)) +
    labs(title='Unknown Starts')
)
#starts.show()

ends = (
    ggplot(map_df) +
    geom_path(aes(x='lon', y='lat', group='group'), color='gray') +
    geom_point(data=unknown_ends, mapping=aes(x='endLon', y='endLat'), color='red', alpha=0.6) +
    coord_fixed() +
    theme_minimal() +
    theme(figure_size=(7, 5)) +
    labs(title='Unknown Ends')
)
#ends.show()

In [ ]:
seqs_per_location_map = (
    ggplot(map_df) +
    geom_path(aes(x='lon', y='lat', group='group'), color='gray') +
    geom_point(data=bubble_data, mapping=aes(x='longitude', y='latitude', size='count', fill = 'flyway'), alpha=0.6) +
    scale_fill_manual(values = {"AtlanticFlyway":"#AA7722", "MississippiFlyway":"#5E9B82", "CentralFlyway":"#C593D1", "PacificFlyway":"#2B57B3", "unknown":"#9C9C9C", "nan":"#9C9C9C"},
                      name = "Flyway",
                      labels = {"AtlanticFlyway":"Atlantic", "MississippiFlyway":"Mississippi", "CentralFlyway":"Central", "PacificFlyway":"Pacific", "unknown":"Unknown", "nan":"Unknown"}) +
    scale_size_continuous(name="Count", range=(3, 15)) +
    coord_fixed() +
    theme(text=element_text(size=24), legend_title=element_text(size=30), legend_text=element_text(size=24), legend_key_size=10) + 
    theme(figure_size=(15, 10)) +
    theme_minimal() +
    labs(title='Sequence Counts by Location')
)